# 🤑 Notebook 02: Algoritmos Voraces (Greedy)
## Algoritmos y Estructuras de Datos
**Universidad de Talca** | Facultad de Ingeniería  
**Docente:** PhD. César Astudillo  
**Semestre:** _________ | **Fecha:** _________

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** el paradigma Greedy: tomar la decisión local óptima en cada paso sin mirar el futuro.
2. **Identificar** cuándo Greedy garantiza la solución óptima y cuándo falla, usando la *propiedad greedy* y la *subestructura óptima*.
3. **Implementar** los algoritmos de Mochila Fraccionaria, Cambio de Monedas y Codificación de Huffman en Python.
4. **Demostrar** con contraejemplos concretos que Greedy puede fallar en variantes del mismo problema.
5. **Comparar** la solución Greedy con la solución óptima exacta para el problema de la Mochila 0/1.

## 🌍 Motivación: La Estrategia del Avaro

Imagina que tienes una mochila con capacidad de **300 kg** y varios objetos que puedes llevar.
Quieres maximizar el valor total. ¿Cómo eliges qué llevar?

**Estrategia intuitiva:** *"Toma siempre lo más valioso disponible."*  
Eso es exactamente un **algoritmo voraz (greedy)**: toma la mejor opción disponible *ahora*, sin planificar el futuro.

Esta estrategia funciona sorprendentemente bien en muchos problemas reales:
- 🗜️ **Compresión de archivos** (Huffman) → ZIP, JPEG, MP3
- 🗺️ **GPS / rutas más cortas** (Dijkstra) → Google Maps, Waze
- 📅 **Planificación de tareas** → sistemas operativos, schedulers
- 🔌 **Redes eléctricas** (Kruskal/Prim) → tendido de fibra óptica

> 📌 **Definición:** Un **algoritmo voraz** (greedy, avaricioso) es un algoritmo de optimización que construye una solución **paso a paso**, eligiendo en cada paso la opción que parece localmente óptima, **sin revisar decisiones pasadas**.

> 🎙️ **[PAUSA PROFESOR]** Pregunta: *"¿Pueden pensar en una situación donde 'tomar lo mejor ahora' lleve a un resultado peor que planificar con cuidado?"*  
> *(Ejemplo: en Tetris, colocar cada pieza en la mejor posición local puede bloquear posiciones futuras.)*

## 📐 Teoría: Cuándo Funciona el Paradigma Greedy

### Los 4 pasos del paradigma (del PDF)

```
Solución ← conjunto vacío
MIENTRAS no sea solución completa:
    candidato ← seleccionar el mejor elemento disponible
    SI candidato es factible (no viola restricciones):
        Solución ← Solución ∪ {candidato}
RETORNAR Solución
```

### Condiciones para que Greedy sea óptimo

Un problema admite solución greedy óptima si tiene:

> 📌 **Propiedad 1 — Elección Greedy:** Siempre existe una solución óptima que incluye la elección greedy (el "mejor" elemento local). Es decir, *nunca nos arrepentimos* de haber tomado lo mejor disponible.

> 📌 **Propiedad 2 — Subestructura Óptima:** Tras tomar la elección greedy, el subproblema restante también tiene estructura óptima. La solución del subproblema se puede combinar con la elección greedy para obtener el óptimo global.

> ⚠️ **Importante:** Si el problema **NO tiene propiedad greedy** (como la Mochila 0/1), Greedy puede dar una solución correcta pero **no necesariamente óptima**. Para esos casos se necesita Programación Dinámica (Notebook 03).

### ¿Greedy o Programación Dinámica?

| | Greedy | Programación Dinámica |
|-|--------|----------------------|
| Decisiones | Irrevocables (no hay vuelta atrás) | Se evalúan todas las opciones |
| Subproblemas | No se generan (la decisión los elimina) | Se conservan en una tabla |
| Complejidad | Generalmente más eficiente | Generalmente más costoso |
| Optimalidad | Solo si hay propiedad greedy | Siempre (si hay subestructura óptima) |

> 💡 **Insight:** La Mochila **Fraccionaria** tiene propiedad greedy → Greedy es óptimo.  
> La Mochila **0/1** (no se puede fraccionar) **no** la tiene → necesita DP.

In [ ]:
# ── Datos unificados de la Mochila (PDF S03) ──────────────────────────────
# IMPORTANTE: estos mismos datos se usan en los Notebooks 02, 03, 04 y 05
# para poder comparar las 4 aproximaciones (Greedy, DP, Backtracking, B&B)
# al mismo problema.

OBJETOS = [
    {'nombre': 'Objeto 1', 'valor': 20, 'peso': 50},
    {'nombre': 'Objeto 2', 'valor': 24, 'peso': 100},
    {'nombre': 'Objeto 3', 'valor': 55, 'peso': 150},
    {'nombre': 'Objeto 4', 'valor': 40, 'peso': 200},
    {'nombre': 'Objeto 5', 'valor': 70, 'peso': 250},
]
CAPACIDAD_W = 300   # capacidad de la mochila en unidades de peso

print("Datos de la mochila (del PDF S03_Diseño_de_Algoritmos):")
print(f"{'Objeto':<12} {'Valor':>8} {'Peso':>8} {'Ratio v/w':>12}")
print("-" * 44)
for obj in OBJETOS:
    ratio = obj['valor'] / obj['peso']
    print(f"{obj['nombre']:<12} {obj['valor']:>8} {obj['peso']:>8} {ratio:>12.4f}")
print(f"\nCapacidad W = {CAPACIDAD_W}")
print(f"Valor total si lleváramos todo: {sum(o['valor'] for o in OBJETOS)}")
print(f"Peso total si lleváramos todo:  {sum(o['peso']  for o in OBJETOS)}")

In [ ]:
# ── Mochila Fraccionaria con 3 criterios Greedy ───────────────────────────
# Demuestra que el criterio ratio v/w es el óptimo para la versión fraccionaria.

def mochila_greedy(objetos: list, capacidad: int, criterio: str,
                   verbose: bool = False) -> dict:
    """
    Resuelve la Mochila Fraccionaria usando el criterio Greedy especificado.
    En la versión fraccionaria se puede llevar una fracción de cada objeto.

    Parámetros:
        objetos (list):  lista de dicts con claves 'nombre', 'valor', 'peso'
        capacidad (int): peso máximo de la mochila
        criterio (str):  'mayor_valor' | 'menor_peso' | 'mayor_ratio'
        verbose (bool):  si True, imprime cada paso de selección

    Retorna:
        dict con claves: 'valor_total', 'peso_usado', 'seleccion'
              donde 'seleccion' es lista de (nombre, fraccion, valor_llevado)

    Complejidad:
        Temporal: O(n log n) — dominado por el ordenamiento
        Espacial: O(n)
    """
    # Función clave de ordenamiento según criterio
    if criterio == 'mayor_valor':
        ordenados = sorted(objetos, key=lambda o: o['valor'], reverse=True)
    elif criterio == 'menor_peso':
        ordenados = sorted(objetos, key=lambda o: o['peso'])
    elif criterio == 'mayor_ratio':
        ordenados = sorted(objetos, key=lambda o: o['valor']/o['peso'], reverse=True)
    else:
        raise ValueError(f"Criterio desconocido: {criterio}")

    peso_restante = capacidad
    valor_total   = 0.0
    seleccion     = []

    for obj in ordenados:
        if peso_restante <= 0:
            break
        # Fracción que podemos llevar (máximo 1.0 = el objeto completo)
        fraccion = min(1.0, peso_restante / obj['peso'])
        valor_llevado = fraccion * obj['valor']
        valor_total  += valor_llevado
        peso_restante -= fraccion * obj['peso']
        seleccion.append((obj['nombre'], fraccion, valor_llevado))

        if verbose:
            pct = f"{fraccion*100:.1f}%"
            print(f"  ✔ {obj['nombre']:10} (v={obj['valor']}, p={obj['peso']}, "
                  f"ratio={obj['valor']/obj['peso']:.4f}) "
                  f"→ llevar {pct} → valor acumulado: {valor_total:.2f}")

    return {
        'valor_total':  round(valor_total, 4),
        'peso_usado':   round(capacidad - peso_restante, 4),
        'seleccion':    seleccion
    }


# ── Comparar los 3 criterios ──────────────────────────────────────────────
criterios = [
    ('mayor_valor',  'Mayor valor primero'),
    ('menor_peso',   'Menor peso primero'),
    ('mayor_ratio',  'Mayor ratio v/w primero (criterio ÓPTIMO)'),
]

for clave, etiqueta in criterios:
    print(f"\n{'='*55}")
    print(f"Criterio: {etiqueta}")
    print('='*55)
    res = mochila_greedy(OBJETOS, CAPACIDAD_W, clave, verbose=True)
    print(f"  → Valor total: {res['valor_total']:.2f} | Peso usado: {res['peso_usado']:.1f}/{CAPACIDAD_W}")

In [ ]:
# Visualización comparativa de los 3 criterios Greedy vs Óptimo 0/1
import matplotlib
import matplotlib.pyplot as plt
import itertools

try:
    import google.colab; EN_COLAB = True
except ImportError:
    EN_COLAB = False
if not EN_COLAB:
    try: get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception: pass

# Calcular óptimo 0/1 por fuerza bruta (n=5, manejable)
def optimo_01_bruta(objetos, capacidad):
    """Calcula el óptimo exacto de la mochila 0/1 por enumeración."""
    n = len(objetos)
    mejor_valor, mejor_sel = 0, []
    for mascara in range(1 << n):
        peso = sum(objetos[i]['peso']  for i in range(n) if mascara >> i & 1)
        val  = sum(objetos[i]['valor'] for i in range(n) if mascara >> i & 1)
        if peso <= capacidad and val > mejor_valor:
            mejor_valor = val
            mejor_sel   = [objetos[i]['nombre'] for i in range(n) if mascara >> i & 1]
    return mejor_valor, mejor_sel

optimo_val, optimo_sel = optimo_01_bruta(OBJETOS, CAPACIDAD_W)

resultados = {}
for clave, etiqueta in criterios:
    r = mochila_greedy(OBJETOS, CAPACIDAD_W, clave)
    resultados[etiqueta] = r['valor_total']

# Añadir óptimo fraccionario y óptimo 0/1
res_ratio = mochila_greedy(OBJETOS, CAPACIDAD_W, 'mayor_ratio')
etiquetas = [e for _, e in criterios] + [f'Óptimo 0/1\n(fuerza bruta)']
valores   = [resultados[e] for _, e in criterios] + [optimo_val]
colores   = ['#F44336', '#FF9800', '#2196F3', '#4CAF50']

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')
barras = ax.bar(etiquetas, valores, color=colores, edgecolor='#212121', width=0.5)
for bar, val in zip(barras, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.axhline(optimo_val, color='#4CAF50', ls='--', lw=1.5, label=f'Óptimo 0/1 = {optimo_val}')
ax.set_ylabel('Valor total obtenido', color='#212121')
ax.set_title(f'Comparación de criterios Greedy (W={CAPACIDAD_W})\n'
             f'Óptimo 0/1: {optimo_sel} → valor {optimo_val}',
             fontweight='bold', color='#212121')
ax.legend(fontsize=9)
ax.set_ylim(0, max(valores) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\n💡 Observa: Greedy con ratio v/w es óptimo para la versión FRACCIONARIA,")
print(f"   pero puede fallar en la versión 0/1 (no se puede fraccionar).")
print(f"   Óptimo 0/1 exacto: {optimo_sel} → valor = {optimo_val}")

In [ ]:
# ── Cambio de Monedas: cuándo Greedy funciona y cuándo falla ──────────────

def cambio_greedy(denominaciones: list, monto: int,
                  verbose: bool = False) -> dict:
    """
    Calcula el cambio para 'monto' usando el mínimo de monedas (algoritmo greedy).
    En cada paso toma la moneda más grande posible.

    Parámetros:
        denominaciones (list): lista de valores de monedas disponibles
        monto (int):           monto a cambiar
        verbose (bool):        si True, imprime cada paso

    Retorna:
        dict: {'monedas_usadas': list, 'total_monedas': int, 'exito': bool}

    Complejidad:
        Temporal: O(n * monto/min_denom) en el peor caso
        Espacial: O(n)
    """
    denoms = sorted(denominaciones, reverse=True)  # mayor primero
    restante = monto
    monedas  = []

    for d in denoms:
        while restante >= d:
            restante -= d
            monedas.append(d)
            if verbose:
                print(f"  Uso moneda {d:3d} → restante: {restante}")

    return {
        'monedas_usadas': monedas,
        'total_monedas':  len(monedas),
        'exito':          restante == 0
    }


# ── Sistema estándar: Greedy es ÓPTIMO ────────────────────────────────────
print("━"*50)
print("Sistema ESTÁNDAR: [25, 10, 5, 1] centavos")
print("━"*50)
for monto in [30, 41, 67]:
    res = cambio_greedy([25, 10, 5, 1], monto, verbose=False)
    print(f"  Monto {monto:3d}: {res['monedas_usadas']} → {res['total_monedas']} monedas")

print()
print("━"*50)
print("Sistema ESPECIAL: [1, 3, 4] centavos — Greedy FALLA")
print("━"*50)
monto_falla = 6
res_greedy = cambio_greedy([1, 3, 4], monto_falla, verbose=True)
print(f"  Greedy → {res_greedy['monedas_usadas']} = {res_greedy['total_monedas']} monedas")
print(f"  Óptimo → [3, 3] = 2 monedas  ← Greedy NO encontró esto!")
print(f"\n⚠️  Para monto={monto_falla}: Greedy usa {res_greedy['total_monedas']} monedas,")
print(f"   pero el óptimo es 2 monedas [3, 3].")
print(f"   Razón: Greedy tomó 4 primero, dejando 2 que requiere 2 monedas de 1.")

In [ ]:
# ── Codificación de Huffman ───────────────────────────────────────────────
# Greedy: selecciona siempre los 2 nodos de menor frecuencia para combinar.
import heapq

class NodoHuffman:
    """Nodo del árbol de Huffman con símbolo, frecuencia e hijos."""
    def __init__(self, simbolo, freq, izq=None, der=None):
        self.simbolo = simbolo
        self.freq    = freq
        self.izq     = izq
        self.der     = der

    # Necesario para que heapq pueda comparar nodos
    def __lt__(self, otro):
        return self.freq < otro.freq


def construir_huffman(frecuencias: dict) -> NodoHuffman:
    """
    Construye el árbol de Huffman usando el algoritmo greedy.
    En cada paso, combina los 2 nodos de menor frecuencia.

    Parámetros:
        frecuencias (dict): {símbolo: frecuencia}

    Retorna:
        NodoHuffman: raíz del árbol de Huffman

    Complejidad:
        Temporal: O(n log n) — n extracciones de min-heap
        Espacial: O(n)
    """
    # Inicializar el heap con un nodo hoja por símbolo
    heap = [NodoHuffman(s, f) for s, f in frecuencias.items()]
    heapq.heapify(heap)

    paso = 0
    print("Construcción del árbol de Huffman (paso a paso):")
    print(f"  Estado inicial: {[(n.simbolo, n.freq) for n in sorted(heap, key=lambda x: x.freq)]}")

    while len(heap) > 1:
        # Greedy: extraer los 2 de menor frecuencia
        nodo1 = heapq.heappop(heap)
        nodo2 = heapq.heappop(heap)
        paso += 1

        # Crear nodo interno combinado
        combinado = NodoHuffman(
            simbolo=f"({nodo1.simbolo}+{nodo2.simbolo})",
            freq=nodo1.freq + nodo2.freq,
            izq=nodo1, der=nodo2
        )
        heapq.heappush(heap, combinado)

        restantes = sorted(heap, key=lambda x: x.freq)
        print(f"  Paso {paso}: combinar ({nodo1.simbolo}, {nodo1.freq}) + "
              f"({nodo2.simbolo}, {nodo2.freq}) → freq={combinado.freq}")
        print(f"     Heap: {[(n.simbolo, n.freq) for n in restantes]}")

    return heap[0]


def generar_codigos(nodo: NodoHuffman, prefijo: str = '') -> dict:
    """Recorre el árbol y genera el código binario para cada símbolo."""
    if nodo.izq is None and nodo.der is None:  # hoja
        return {nodo.simbolo: prefijo or '0'}   # símbolo único → código '0'
    codigos = {}
    if nodo.izq:  codigos.update(generar_codigos(nodo.izq, prefijo + '0'))
    if nodo.der:  codigos.update(generar_codigos(nodo.der, prefijo + '1'))
    return codigos


# ── Demo con frecuencias del PDF ──────────────────────────────────────────
# Símbolos a-f con frecuencias proporcionales
FRECUENCIAS = {'a': 45, 'b': 13, 'c': 12, 'd': 16, 'e': 9, 'f': 5}
print(f"Frecuencias de entrada: {FRECUENCIAS}\n")
raiz = construir_huffman(FRECUENCIAS)
codigos = generar_codigos(raiz)

total_frec = sum(FRECUENCIAS.values())
bits_huffman = sum(FRECUENCIAS[s] * len(c) for s, c in codigos.items())
bits_fijos   = total_frec * 3  # 3 bits por símbolo con codificación fija

print("\nCódigos de Huffman generados:")
print(f"  {'Símbolo':>8} {'Freq':>6} {'Código':>10} {'Bits usados':>12}")
print("  " + "-" * 42)
for s, c in sorted(codigos.items(), key=lambda x: FRECUENCIAS[x[0]], reverse=True):
    print(f"  {s:>8} {FRECUENCIAS[s]:>6} {c:>10} {FRECUENCIAS[s]*len(c):>12}")
print(f"\n  Bits totales con Huffman:        {bits_huffman}")
print(f"  Bits totales con código fijo(3): {bits_fijos}")
print(f"  Ahorro: {bits_fijos - bits_huffman} bits ({(bits_fijos-bits_huffman)/bits_fijos*100:.1f}% compresión)")

In [ ]:
# ── Dijkstra: ruta más corta (algoritmo Greedy clásico) ──────────────────
# En cada paso elige el nodo no visitado de menor distancia acumulada.
import heapq

def dijkstra(grafo: dict, origen: str, verbose: bool = False) -> dict:
    """
    Calcula las distancias más cortas desde 'origen' a todos los nodos.
    Greedy: en cada paso expande el nodo más cercano aún no visitado.

    Parámetros:
        grafo (dict):  {nodo: [(vecino, peso), ...]} lista de adyacencia
        origen (str):  nodo de partida
        verbose (bool): si True, imprime cada expansión

    Retorna:
        dict: {nodo: distancia_minima_desde_origen}

    Complejidad:
        Temporal: O((V + E) log V) con min-heap
        Espacial: O(V)
    """
    distancias = {nodo: float('inf') for nodo in grafo}
    distancias[origen] = 0
    heap = [(0, origen)]  # (distancia, nodo)
    visitados = set()
    paso = 0

    while heap:
        dist_actual, nodo = heapq.heappop(heap)
        if nodo in visitados:
            continue
        visitados.add(nodo)
        paso += 1
        if verbose:
            print(f"  Paso {paso}: expandir '{nodo}' (dist={dist_actual})")

        # Greedy: explorar vecinos y actualizar distancias
        for vecino, peso in grafo.get(nodo, []):
            nueva_dist = dist_actual + peso
            if nueva_dist < distancias[vecino]:
                distancias[vecino] = nueva_dist
                heapq.heappush(heap, (nueva_dist, vecino))
                if verbose:
                    print(f"    Actualizar dist['{vecino}'] = {nueva_dist}")

    return distancias


# ── Demo: grafo pequeño de 6 nodos ────────────────────────────────────────
GRAFO = {
    'A': [('B', 4), ('C', 2)],
    'B': [('A', 4), ('C', 1), ('D', 5)],
    'C': [('A', 2), ('B', 1), ('D', 8), ('E', 10)],
    'D': [('B', 5), ('C', 8), ('E', 2), ('F', 6)],
    'E': [('C', 10), ('D', 2), ('F', 3)],
    'F': [('D', 6), ('E', 3)],
}

print("Grafo: A→B(4), A→C(2), B→C(1), B→D(5), C→D(8), C→E(10), D→E(2), D→F(6), E→F(3)")
print("\n🔍 Ejecución de Dijkstra desde 'A':")
distancias = dijkstra(GRAFO, 'A', verbose=True)
print("\nDistancias mínimas desde A:")
for nodo, dist in sorted(distancias.items()):
    print(f"  A → {nodo}: {dist}")
print("\n💡 Dijkstra es Greedy: siempre expande el nodo más cercano aún no visitado.")
print("   Esto funciona porque los pesos son NO negativos.")

In [ ]:
# Animación: construcción Greedy de la mochila fraccionaria
# Muestra barra a barra cómo se llena la mochila con el criterio ratio v/w
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from IPython.display import HTML, display

C_ACTIVO    = '#2196F3'  # azul — objeto siendo procesado
C_CANDIDATO = '#FF9800'  # naranja — siguiente candidato
C_OK        = '#4CAF50'  # verde — seleccionado
C_DESCART   = '#F44336'  # rojo — no cabe
C_FONDO     = '#FAFAFA'
C_TEXTO     = '#212121'

def animar_mochila_greedy(objetos, capacidad):
    """Genera animación de la selección greedy (criterio mayor_ratio)."""
    ordenados = sorted(objetos, key=lambda o: o['valor']/o['peso'], reverse=True)
    n = len(ordenados)

    # Pre-calcular frames: (paso, objeto_idx, fraccion, peso_acum, val_acum, estado)
    # estado: 'candidato', 'seleccionado', 'parcial', 'descartado'
    frames = []
    peso_acum = 0.0
    val_acum  = 0.0
    estados   = ['pendiente'] * n
    fracciones = [0.0] * n

    for idx, obj in enumerate(ordenados):
        estados_frame = estados.copy()
        estados_frame[idx] = 'candidato'
        frames.append((idx, estados_frame.copy(), fracciones.copy(), peso_acum, val_acum))

        if peso_acum >= capacidad:
            estados[idx] = 'descartado'
            continue
        frac = min(1.0, (capacidad - peso_acum) / obj['peso'])
        fracciones[idx] = frac
        peso_acum += frac * obj['peso']
        val_acum  += frac * obj['valor']
        estados[idx] = 'seleccionado' if frac == 1.0 else 'parcial'
        frames.append((idx, estados.copy(), fracciones.copy(), peso_acum, val_acum))

    fig, (ax_obj, ax_moch) = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(C_FONDO)
    for ax in (ax_obj, ax_moch):
        ax.set_facecolor(C_FONDO)

    nombres = [o['nombre'] for o in ordenados]
    ratios  = [o['valor']/o['peso'] for o in ordenados]

    # Panel izq: objetos disponibles con ratio
    ax_obj.set_xlim(-0.5, n - 0.5)
    ax_obj.set_ylim(0, max(ratios) * 1.3)
    ax_obj.set_xticks(range(n))
    ax_obj.set_xticklabels(nombres, fontsize=8)
    ax_obj.set_ylabel('Ratio v/w', color=C_TEXTO)
    ax_obj.set_title('Objetos (ordenados por ratio v/w)', fontweight='bold', color=C_TEXTO)
    barras_obj = ax_obj.bar(range(n), ratios, color='#90CAF9', edgecolor=C_TEXTO, lw=0.8)
    for i, (bar, obj) in enumerate(zip(barras_obj, ordenados)):
        ax_obj.text(i, ratios[i] + 0.005, f"v={obj['valor']}\np={obj['peso']}",
                    ha='center', va='bottom', fontsize=7, color=C_TEXTO)

    # Panel der: mochila llenándose
    ax_moch.set_xlim(0, capacidad * 1.05)
    ax_moch.set_ylim(-0.5, n - 0.5)
    ax_moch.set_xlabel('Peso acumulado', color=C_TEXTO)
    ax_moch.axvline(capacidad, color='#F44336', ls='--', lw=2, label=f'W={capacidad}')
    ax_moch.set_title('Mochila llenándose', fontweight='bold', color=C_TEXTO)
    ax_moch.legend(fontsize=9)
    barras_moch = [ax_moch.barh(i, 0, height=0.6,
                                 left=sum(ordenados[j]['peso']*fracciones[j]
                                          if j < i else 0 for j in range(n)),
                                 color=C_OK, edgecolor=C_TEXTO, lw=0.8)[0]
                   for i in range(n)]
    titulo_moch = ax_moch.set_title('Mochila llenándose', fontweight='bold', color=C_TEXTO)
    info_txt = ax_moch.text(0.98, 0.02, '', transform=ax_moch.transAxes,
                             ha='right', va='bottom', fontsize=9, color=C_TEXTO)

    COLOR_MAP = {'pendiente': '#90CAF9', 'candidato': C_CANDIDATO,
                 'seleccionado': C_OK, 'parcial': '#A5D6A7', 'descartado': C_DESCART}

    def actualizar(fi):
        idx_frame, estados_f, fracs_f, peso_f, val_f = frames[fi]
        for i, (bar, est) in enumerate(zip(barras_obj, estados_f)):
            bar.set_facecolor(COLOR_MAP.get(est, '#90CAF9'))

        # Recalcular barras mochila
        acum = 0
        for i, (bar, obj, frac) in enumerate(zip(barras_moch, ordenados, fracs_f)):
            w = frac * obj['peso']
            bar.set_x(acum); bar.set_width(w)
            bar.set_facecolor(COLOR_MAP.get(estados_f[i], '#90CAF9'))
            ax_moch.text(acum + w/2, i,
                         f"{frac*100:.0f}%" if frac > 0 else '',
                         ha='center', va='center', fontsize=7.5,
                         color='white', fontweight='bold')
            acum += w

        info_txt.set_text(f'Peso: {peso_f:.0f}/{capacidad}\nValor: {val_f:.1f}')
        return barras_obj + barras_moch

    anim = animation.FuncAnimation(fig, actualizar, frames=len(frames),
                                    interval=800, repeat=False, blit=False)
    plt.tight_layout()
    plt.close(fig)
    return anim

anim = animar_mochila_greedy(OBJETOS, CAPACIDAD_W)
display(HTML(anim.to_jshtml()))

## 📈 Análisis de Complejidad

| Algoritmo | Complejidad Temporal | Complejidad Espacial | Óptimo |
|-----------|---------------------|---------------------|--------|
| Mochila Fraccionaria | $O(n \log n)$ — ordenar + $O(n)$ selección | $O(n)$ | ✅ Sí |
| Mochila 0/1 con Greedy | $O(n \log n)$ | $O(n)$ | ❌ No siempre |
| Cambio de Monedas (estándar) | $O(n \cdot \lfloor M/d_{min} \rfloor)$ | $O(M)$ | ✅ Sí (estándar) |
| Huffman | $O(n \log n)$ — $n$ operaciones de min-heap | $O(n)$ | ✅ Óptimo compresión |
| Dijkstra (min-heap) | $O((V + E) \log V)$ | $O(V)$ | ✅ Pesos no negativos |

> ⚠️ **Importante:** Dijkstra **falla** si hay pesos negativos en el grafo. Para ese caso se usa el algoritmo de Bellman-Ford ($O(VE)$), que no es Greedy.

> 💡 **Insight:** Greedy es generalmente más rápido que DP, pero exige una **demostración** de que la propiedad greedy se cumple. Nunca apliques Greedy sin verificar esto — puedes obtener soluciones incorrectas silenciosamente.

In [ ]:
# Widget interactivo: Explorador de la Mochila Greedy
# Permite editar objetos, cambiar W y criterio, y comparar con fuerza bruta.
# Auto-contenido — no necesita celdas anteriores.
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# ── Datos iniciales (del PDF) ─────────────────────────────────────────────
OBJS_INIT = [
    {'nombre': 'Obj 1', 'valor': 20, 'peso': 50},
    {'nombre': 'Obj 2', 'valor': 24, 'peso': 100},
    {'nombre': 'Obj 3', 'valor': 55, 'peso': 150},
    {'nombre': 'Obj 4', 'valor': 40, 'peso': 200},
    {'nombre': 'Obj 5', 'valor': 70, 'peso': 250},
]

def _optimo_01(objetos, cap):
    """Óptimo exacto 0/1 por fuerza bruta (n≤8)."""
    n = len(objetos); best = 0
    for m in range(1 << n):
        p = sum(objetos[i]['peso']  for i in range(n) if m >> i & 1)
        v = sum(objetos[i]['valor'] for i in range(n) if m >> i & 1)
        if p <= cap: best = max(best, v)
    return best

def _greedy(objetos, cap, criterio):
    """Greedy 0/1 (no fraccionario) para comparar con óptimo."""
    if criterio == 'valor':  key = lambda o: o['valor']
    elif criterio == 'peso': key = lambda o: -o['peso']
    else:                    key = lambda o: o['valor']/max(o['peso'],1)
    ordenados = sorted(objetos, key=key, reverse=True)
    peso_r, val, sel = cap, 0, []
    for o in ordenados:
        if o['peso'] <= peso_r:
            peso_r -= o['peso']; val += o['valor']; sel.append(o['nombre'])
    return val, sel

# ── Controles ─────────────────────────────────────────────────────────────
n_slider   = widgets.IntSlider(value=5, min=2, max=8, description='# objetos:',
                                style={'description_width':'initial'},
                                layout=widgets.Layout(width='350px'))
w_slider   = widgets.IntSlider(value=300, min=50, max=600, step=25,
                                description='Capacidad W:',
                                style={'description_width':'initial'},
                                layout=widgets.Layout(width='350px'))
crit_sel   = widgets.Dropdown(
    options=[('Mayor ratio v/w (óptimo fraccionario)', 'ratio'),
             ('Mayor valor primero', 'valor'),
             ('Menor peso primero', 'peso')],
    description='Criterio greedy:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='380px')
)
boton  = widgets.Button(description='▶ Calcular', button_style='primary')
salida = widgets.Output()

def al_calcular(b):
    with salida:
        salida.clear_output(wait=True)
        n   = n_slider.value
        cap = w_slider.value
        crt = crit_sel.value

        # Usar los primeros n objetos de la lista inicial
        # (en un widget más complejo podrían ser editables)
        objs = OBJS_INIT[:n]

        val_greedy, sel_greedy = _greedy(objs, cap, crt)
        val_optimo = _optimo_01(objs, cap)
        gap = val_optimo - val_greedy
        es_optimo = (gap == 0)

        etiqueta_crit = {v: l for l, v in crit_sel.options}.get(crt, crt)
        print(f"Objetos: {[o['nombre'] for o in objs]}")
        print(f"Capacidad W = {cap}")
        print(f"Criterio: {etiqueta_crit}")
        print()
        print(f"  Greedy 0/1 → selección: {sel_greedy}")
        print(f"             → valor obtenido: {val_greedy}")
        print(f"  Óptimo 0/1 → valor óptimo:   {val_optimo}")
        if es_optimo:
            print(f"  ✅ ¡Greedy encontró el óptimo en este caso!")
        else:
            print(f"  ❌ Greedy NO es óptimo — gap = {gap} ({gap/val_optimo*100:.1f}% peor)")

        fig, ax = plt.subplots(figsize=(6, 3.5))
        fig.patch.set_facecolor('#FAFAFA'); ax.set_facecolor('#FAFAFA')
        ax.bar(['Greedy 0/1', 'Óptimo 0/1 (exacto)'],
               [val_greedy, val_optimo],
               color=['#FF9800' if not es_optimo else '#4CAF50', '#4CAF50'],
               edgecolor='#212121', width=0.4)
        ax.set_ylabel('Valor total', color='#212121')
        ax.set_title(f'Greedy vs Óptimo (W={cap}, {n} objetos)', fontweight='bold')
        ax.set_ylim(0, val_optimo * 1.25)
        for i, v in enumerate([val_greedy, val_optimo]):
            ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

boton.on_click(al_calcular)
display(widgets.VBox([
    widgets.HTML('<h4 style="color:#212121">🎒 Explorador de la Mochila Greedy</h4>'),
    widgets.HTML('<em style="color:#555">Usa los primeros N objetos del PDF. '
                 'Cambia el criterio para ver cuándo Greedy falla.</em>'),
    n_slider, w_slider, crit_sel, boton, salida
]))

## 🧪 Ejercicio 1: Activity Selection Problem ⭐

**Descripción:** Dado un conjunto de tareas con tiempos de **inicio** y **fin**, selecciona el
**máximo número de tareas** que se pueden realizar sin solapamiento. Solo puedes hacer una tarea
a la vez, y cada tarea ocupa el intervalo $[inicio, fin)$.

Este es el ejemplo clásico donde Greedy **sí** es óptimo. El criterio: ordenar por tiempo de
**fin** (no por inicio ni por duración) y tomar la tarea que termina antes.

**Entrada:** lista de tuplas `(inicio, fin)`

**Salida:** lista de tareas seleccionadas + cantidad total

**Ejemplo:**
```
Entrada: [(1,3),(2,5),(4,7),(6,9),(5,8),(8,10)]
Salida:  [(1,3),(4,7),(8,10)] — 3 tareas
```

**Restricciones:** $1 \leq n \leq 10^5$, $0 \leq inicio < fin \leq 10^6$  
**Complejidad esperada:** $O(n \log n)$

In [ ]:
def activity_selection(tareas: list) -> list:
    """
    Selecciona el máximo número de tareas sin solapamiento (Activity Selection Problem).
    Criterio Greedy: ordenar por tiempo de FIN y tomar la tarea que termina antes.

    Parámetros:
        tareas (list): lista de tuplas (inicio, fin)

    Retorna:
        list: lista de tuplas (inicio, fin) de las tareas seleccionadas

    Complejidad:
        Temporal: O(n log n) — ordenamiento + O(n) selección lineal
        Espacial: O(n)

    Ejemplo:
        >>> activity_selection([(1,3),(2,5),(4,7),(6,9)])
        [(1,3),(4,7),(6,9)]
    """
    # Tu código aquí
    # Pista: ordena por el SEGUNDO elemento de cada tupla (tiempo de fin).
    # Luego, selecciona una tarea si su inicio >= fin de la última seleccionada.
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Ejecuta casos de prueba para activity_selection."""
    import time
    casos = [
        (([],),                       [],              'Lista vacía'),
        (([(1, 5)],),                 [(1, 5)],        'Una sola tarea'),
        (([(1, 3), (2, 5), (4, 7), (6, 9)],),
          [(1, 3), (4, 7), (6, 9)],   'Ejemplo del enunciado — 3 tareas'),
        (([(1, 10), (2, 3), (4, 5), (6, 7)],),
          [(2, 3), (4, 5), (6, 7)],   'Tarea larga no es óptima — 3 tareas cortas'),
        (([(1, 2), (2, 3), (3, 4), (4, 5)],),
          [(1, 2), (2, 3), (3, 4), (4, 5)], 'Tareas consecutivas — todas se pueden hacer'),
        (([(1, 5), (1, 5), (1, 5)],),
          [(1, 5)],                    'Todas iguales — solo 1'),
    ]
    aprobados = 0
    for args, esperado, descripcion in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(*args)
            t1 = time.perf_counter()
            # Verificar cantidad y validez (sin solapamientos)
            if len(resultado) == len(esperado):
                print(f"  ✅ {descripcion} — {len(resultado)} tareas ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {descripcion}")
                print(f"     Esperado: {len(esperado)} tareas = {esperado}")
                print(f"     Obtenido: {len(resultado)} tareas = {resultado}")
        except Exception as e:
            print(f"  💥 {descripcion} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(activity_selection)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def activity_selection(tareas):
#     """Greedy óptimo: ordenar por fin, tomar la que termina antes sin solapamiento."""
#     if not tareas:
#         return []
#     # Paso 1: ordenar por tiempo de FIN (criterio clave)
#     ordenadas = sorted(tareas, key=lambda t: t[1])
#     # Paso 2: tomar la primera tarea siempre
#     seleccionadas = [ordenadas[0]]
#     fin_ultima = ordenadas[0][1]
#     # Paso 3: greedy — agregar tarea si su inicio >= fin de la última
#     for inicio, fin in ordenadas[1:]:
#         if inicio >= fin_ultima:   # no hay solapamiento
#             seleccionadas.append((inicio, fin))
#             fin_ultima = fin
#     return seleccionadas
#
# # ¿Por qué ordenar por FIN y no por INICIO o DURACIÓN?
# # Porque tomar la tarea que termina ANTES libera el máximo tiempo disponible
# # para las tareas futuras. Es la propiedad greedy de este problema.

## 🧪 Ejercicio 2: Cambio de Monedas ⭐⭐

**Descripción:** Dado un sistema de monedas y un monto objetivo, encuentra el cambio
usando el **menor número de monedas** posible con el algoritmo Greedy.

Implementa la función y pruébala con **dos sistemas** de monedas:  
- Sistema estándar: `[1, 5, 10, 25]` → Greedy **sí** es óptimo  
- Sistema especial: `[1, 3, 4]` → Greedy **falla** para ciertos montos

**Entrada:** lista de denominaciones + monto objetivo  
**Salida:** lista de monedas usadas (o lista vacía si no es posible)

**Ejemplo:**
```
Entrada: denominaciones=[1,3,4], monto=6
Salida greedy:  [4, 1, 1]  → 3 monedas  ← subóptimo
Salida óptima:  [3, 3]     → 2 monedas
```

**Complejidad esperada:** $O(n \cdot M)$ donde $n$ = número de denominaciones, $M$ = monto

In [ ]:
def cambio_monedas(denominaciones: list, monto: int) -> list:
    """
    Calcula el cambio para 'monto' usando el algoritmo Greedy.
    En cada paso toma la moneda más grande posible.

    Parámetros:
        denominaciones (list): denominaciones de monedas disponibles
        monto (int):           monto a cambiar (>= 0)

    Retorna:
        list: monedas usadas (puede ser subóptima para ciertos sistemas)

    Complejidad:
        Temporal: O(n * monto / d_min)
        Espacial: O(monto / d_min) en el peor caso
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Ejecuta casos de prueba para cambio_monedas."""
    import time
    casos = [
        # Sistema estándar — Greedy es óptimo
        (([25,10,5,1], 30),  [25, 5],        'Estándar: 30 centavos'),
        (([25,10,5,1], 41),  [25, 10, 5, 1], 'Estándar: 41 centavos'),
        (([25,10,5,1], 0),   [],              'Monto cero'),
        (([25,10,5,1], 25),  [25],            'Estándar: exacto 25'),
        # Para el sistema especial verificamos solo la cantidad, no la lista exacta
    ]
    aprobados = 0
    for args, esperado, descripcion in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(*args)
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {descripcion} — {resultado} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {descripcion}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {descripcion} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")
    print("\n── Demostración del fallo de Greedy con sistema [1,3,4] ──")
    for monto in [6, 7, 9]:
        try:
            res_greedy = fn([1,3,4], monto)
            print(f"  Monto {monto}: Greedy → {res_greedy} ({len(res_greedy)} monedas)")
        except Exception as e:
            print(f"  Monto {monto}: Error — {e}")
    print("  Monto 6: Óptimo → [3, 3] (2 monedas) — Greedy usa 3")
    print("  Monto 7: Óptimo → [4, 3] (2 monedas) — verifica si Greedy coincide")

verificar_ejercicio_2(cambio_monedas)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def cambio_monedas(denominaciones, monto):
#     """Cambio de monedas greedy — toma siempre la moneda más grande posible."""
#     denoms = sorted(denominaciones, reverse=True)
#     restante = monto
#     monedas  = []
#     for d in denoms:
#         while restante >= d:
#             restante -= d
#             monedas.append(d)
#     return monedas  # puede ser incompleto si restante > 0
#
# # ¿Por qué falla con [1,3,4] y monto=6?
# # Greedy toma 4 → restante=2 → toma 1, 1 → [4,1,1] = 3 monedas
# # Óptimo: [3,3] = 2 monedas. Greedy no "mira" que 3+3=6 es posible.
# # Para el óptimo exacto se necesita Programación Dinámica (Notebook 03).

## 🧪 Ejercicio 3: Scheduling con Penalidades ⭐⭐⭐

**Descripción:** Tienes $n$ trabajos, cada uno con una **ganancia** $g_i$ y un **plazo** $d_i$
(el trabajo debe completarse en la unidad de tiempo $d_i$ o antes, de lo contrario se pierde la ganancia).
Cada trabajo toma exactamente **1 unidad de tiempo**. Solo puedes hacer **un trabajo por unidad de tiempo**.
Selecciona un subconjunto de trabajos que maximice la ganancia total.

**Criterio Greedy óptimo:** ordenar por ganancia descendente y asignar al slot de tiempo más
tardío disponible antes del plazo.

**Entrada:** lista de tuplas `(ganancia, plazo)`  
**Salida:** lista de trabajos seleccionados + ganancia total

**Ejemplo:**
```
Entrada: [(100,2),(19,1),(27,2),(25,1),(15,3)]
Salida:  trabajos seleccionados con ganancia total = 142
         (posible selección: ganancias 100, 27, 15)
```

**Complejidad esperada:** $O(n^2)$ para la versión simple (hay versión $O(n \log n)$ con Union-Find)  
**Nivel CF equivalente:** ~1200

In [ ]:
def job_scheduling(trabajos: list) -> tuple:
    """
    Selecciona trabajos para maximizar ganancia total respetando plazos.
    Cada trabajo toma 1 unidad de tiempo y tiene un plazo y ganancia.

    Parámetros:
        trabajos (list): lista de tuplas (ganancia, plazo)

    Retorna:
        tuple: (ganancia_total, lista_de_trabajos_seleccionados)

    Complejidad:
        Temporal: O(n²) — para cada trabajo busca slot disponible
        Espacial: O(n)

    Ejemplo:
        >>> job_scheduling([(100,2),(19,1),(27,2),(25,1),(15,3)])
        (142, [(100,2),(27,2),(15,3)])
    """
    # Tu código aquí
    # Pista:
    # 1. Ordenar por ganancia descendente.
    # 2. Crear array de slots de tiempo (inicialmente todos libres).
    # 3. Para cada trabajo (en orden de ganancia), busca el slot más tardío
    #    disponible en [1..plazo]. Si existe, asignarlo.
    pass

In [ ]:
def verificar_ejercicio_3(fn):
    """Ejecuta casos de prueba para job_scheduling."""
    import time
    casos = [
        (([(100,2),(19,1),(27,2),(25,1),(15,3)],), 142, 'Ejemplo del enunciado'),
        (([(20,1),(15,1),(10,1)],),                 20, 'Todos mismo plazo=1 — solo 1 cabe'),
        (([(50,1),(40,2),(30,3)],),                120, 'Todos distintos plazos — todos caben'),
        (([(10,2),(5,1)],),                         15, 'Dos trabajos que caben'),
        (([(100,1),(80,1),(60,1)],),               100, 'Plazo 1 para todos — solo el de mayor ganancia'),
    ]
    aprobados = 0
    for args, esperado, descripcion in casos:
        t0 = time.perf_counter()
        try:
            ganancia, seleccion = fn(*args)
            t1 = time.perf_counter()
            if ganancia == esperado:
                print(f"  ✅ {descripcion} — ganancia={ganancia} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {descripcion}")
                print(f"     Esperado ganancia: {esperado}")
                print(f"     Obtenido ganancia: {ganancia} (selección: {seleccion})")
        except Exception as e:
            print(f"  💥 {descripcion} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_3(job_scheduling)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def job_scheduling(trabajos):
#     """Greedy de scheduling: ganancia máxima respetando plazos — O(n²)."""
#     # Paso 1: ordenar por ganancia descendente
#     ordenados = sorted(trabajos, key=lambda t: t[0], reverse=True)
#     max_plazo  = max(t[1] for t in trabajos)
#
#     # Paso 2: array de slots (None = libre, (g,d) = ocupado)
#     slots = [None] * (max_plazo + 1)  # índice 0 no se usa
#
#     seleccionados = []
#     ganancia_total = 0
#
#     for g, d in ordenados:
#         # Paso 3: buscar el slot más tardío disponible en [1..d]
#         for t in range(min(d, max_plazo), 0, -1):
#             if slots[t] is None:
#                 slots[t] = (g, d)
#                 seleccionados.append((g, d))
#                 ganancia_total += g
#                 break
#
#     return ganancia_total, seleccionados
#
# # ¿Por qué funciona? Propiedad greedy: asignar el trabajo más valioso primero
# # y al slot más tardío posible maximiza el espacio para trabajos futuros.
# # Complejidad: O(n²) — puede mejorarse a O(n log n) con Union-Find.

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Qué pasa con la Mochila Greedy si aumentas la capacidad W a 500? ¿Y a 750?
- ¿Puedes encontrar un sistema de monedas donde Greedy falle para **todos** los montos impares?
- Implementa Huffman para un texto en español y calcula la tasa de compresión real.

In [ ]:
# Espacio libre para experimentar
# Sugerencia: prueba mochila_greedy con distintas capacidades (100, 200, 300, 400, 500)
# y grafica el valor obtenido por cada criterio

In [ ]:
# Espacio libre para experimentar
# Sugerencia: usa construir_huffman con las frecuencias de letras de un texto real en español

In [ ]:
# Autoevaluación — 4 preguntas sobre Algoritmos Voraces
import ipywidgets as widgets
from IPython.display import display

preguntas = [
    {
        'pregunta': '1. ¿Cuál es el criterio Greedy ÓPTIMO para la Mochila Fraccionaria?',
        'opciones': [
            'Tomar primero el objeto de mayor valor absoluto',
            'Tomar primero el objeto de menor peso',
            'Tomar primero el objeto de mayor ratio valor/peso',
            'Cualquier criterio funciona igual de bien'
        ],
        'correcta': 2,
        'explicacion': 'El ratio v/w mide la "eficiencia" de cada objeto. Tomar primero los más eficientes maximiza el valor por unidad de peso. Esta es la propiedad greedy de la mochila fraccionaria.'
    },
    {
        'pregunta': '2. ¿Por qué Greedy con ratio v/w NO funciona para la Mochila 0/1?',
        'opciones': [
            'Porque los objetos en 0/1 no tienen ratio calculable',
            'Porque en 0/1 no se puede fraccionar: tomar un objeto "desalineado" puede desperdiciar capacidad',
            'Porque Greedy siempre falla en problemas de mochila',
            'Porque el ratio cambia dinámicamente al llenar la mochila'
        ],
        'correcta': 1,
        'explicacion': 'En la versión 0/1, llevar un objeto grande puede dejar espacio residual que no puede llenarse eficientemente. Greedy no "ve" esta consecuencia. Necesitamos DP para evaluar todas las combinaciones.'
    },
    {
        'pregunta': '3. ¿Por qué Huffman es un algoritmo Greedy?',
        'opciones': [
            'Porque usa un árbol binario',
            'Porque en cada paso selecciona los 2 nodos de MENOR frecuencia para combinar',
            'Porque siempre produce el árbol más balanceado',
            'Porque trabaja sobre el problema completo de una vez'
        ],
        'correcta': 1,
        'explicacion': 'Huffman es Greedy porque en cada paso toma la decisión local óptima: combinar los 2 nodos de menor frecuencia. Esto garantiza que los símbolos más frecuentes tengan los códigos más cortos.'
    },
    {
        'pregunta': '4. En Activity Selection Problem, ¿por qué ordenar por tiempo de FIN es mejor que ordenar por DURACIÓN?',
        'opciones': [
            'No hay diferencia, ambos criterios dan el mismo resultado',
            'Ordenar por duración es el criterio correcto, no por fin',
            'Ordenar por fin libera el máximo tiempo posible para actividades futuras',
            'Ordenar por inicio es el criterio óptimo'
        ],
        'correcta': 2,
        'explicacion': 'Una actividad que termina antes deja más tiempo disponible para elegir actividades futuras. Ordenar por duración puede seleccionar una actividad corta pero que empieza tarde, bloqueando muchas otras.'
    }
]

def crear_quiz(preguntas):
    for i, p in enumerate(preguntas):
        radio  = widgets.RadioButtons(options=p['opciones'], description=f'P{i+1}:',
                                       style={'description_width':'initial'},
                                       layout={'width':'max-content'})
        boton  = widgets.Button(description='Verificar', button_style='info')
        salida = widgets.Output()
        label  = widgets.HTML(f'<b>{p["pregunta"]}</b>')
        def verificar(b, r=radio, out=salida,
                      c=p['correcta'], ex=p['explicacion'], opts=p['opciones']):
            with out:
                out.clear_output()
                if r.value == opts[c]:
                    print(f'✅ ¡Correcto! {ex}')
                else:
                    print(f'❌ No exactamente. Respuesta: "{opts[c]}"\n   {ex}')
        boton.on_click(verificar)
        display(widgets.VBox([label, radio, boton, salida]))
        print('─' * 65)

crear_quiz(preguntas)

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) | 4ª ed. | **Cap. 15** | Greedy Algorithms — Activity Selection (15.1), Huffman (15.3) |
| Kleinberg & Tardos | 1ª ed. | **Cap. 4** | Greedy Algorithms — Scheduling, Dijkstra, Huffman |
| Skiena | 3ª ed. | **Cap. 8** | Weighted Graph Algorithms — Dijkstra, Prim, Kruskal |

### Recursos Gratuitos

- 🌐 [VisuAlgo — Greedy](https://visualgo.net/en/mst) — visualización de Kruskal y Prim
- 📖 [CP-Algorithms — Greedy](https://cp-algorithms.com/graph/dijkstra.html) — Dijkstra con código
- 🌐 [USACO Guide — Greedy](https://usaco.guide/bronze/intro-greedy?lang=py) — guía con Python 3

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** [codeforces.com/problemset](https://codeforces.com/problemset) → Tag: `greedy`

| # | Criterio de búsqueda | Rating | Por qué es útil |
|---|---------------------|--------|----------------|
| 1 | Tag `greedy` + Rating 800 | ⭐ 800 | Aplicación directa — hay >200 problemas en este rango |
| 2 | Tag `greedy` + Rating 1000 | ⭐⭐ 1000 | Requiere identificar el criterio greedy correcto |
| 3 | Tag `greedy` + Tag `sortings` + Rating 1200 | ⭐⭐⭐ 1200 | Combina ordenamiento con decisión greedy |
| 4 | Tag `greedy` + Tag `dp` + Rating 1400 | 🏆 1400+ | Problemas donde Greedy y DP se combinan |

> ⚠️ Los problemas 1 y 2 son el **mínimo esperado**. El tag `greedy` tiene más de 500 problemas en CF — es el paradigma más representado.

---

**Próximo notebook:** [03_programacion_dinamica.ipynb](03_programacion_dinamica.ipynb) — cuando Greedy no alcanza, DP al rescate.